In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy import stats
from scipy.special import expit
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from notebooks.imports import *
from config import dir_config

plt.rcParams.update({'font.family': 'sans-serif', 'axes.grid': False})
DPI = 150

## Load Results

In [ ]:
processed_dir = Path(dir_config.data.processed)
glm_hmm_dir   = Path(processed_dir, 'glm_hmm_models')
fig_dir       = Path(glm_hmm_dir, 'figures')
fig_dir.mkdir(exist_ok=True)

MODEL_NAME = 'prior_model_color_1back'

with open(Path(glm_hmm_dir, f'{MODEL_NAME}_config.pkl'), 'rb') as f:
    feature_config = pickle.load(f)
MODEL_FEATURES = feature_config['model_features']
INPUT_DIM      = feature_config['input_dim']
subject_pairs  = feature_config['subject_pairs']

with open(Path(glm_hmm_dir, f'{MODEL_NAME}_hc.pkl'), 'rb') as f:
    hc_res = pickle.load(f)
with open(Path(glm_hmm_dir, f'{MODEL_NAME}_pd_groups.pkl'), 'rb') as f:
    pd_res = pickle.load(f)
with open(Path(glm_hmm_dir, f'{MODEL_NAME}_analysis.pkl'), 'rb') as f:
    analysis = pickle.load(f)

CONSENSUS_K  = analysis['consensus_k']
all_data     = analysis['all_data']
df_weights   = analysis['df_weights']
GROUP_LABELS = analysis['group_labels']
GROUP_ORDER  = analysis['group_order']
GROUP_COLORS = analysis['group_colors']
boot_results = analysis['boot_results']

STATE_RANGE  = np.arange(1, 5)
STATE_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:CONSENSUS_K]

# Axes use direction-normalised framing
STIM_LABEL   = 'Signed Coherence (+ = toward most frequent direction)'
CHOICE_LABEL = 'P(choice toward most frequent direction)'

print(f'Consensus K = {CONSENSUS_K}')
print(f'Model features: {MODEL_FEATURES}')

## Figure 1 — Cross-Group Model Selection

Test LL vs K for all 5 groups on one panel. Vertical line marks the consensus K.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=DPI)

# HC
hc_ll   = hc_res['cv']['test_ll']
hc_mean = np.nanmean(hc_ll, axis=(0, 2))
hc_sem  = np.nanstd(hc_ll, axis=(0, 2)) / np.sqrt(hc_ll.shape[0] * hc_ll.shape[2])
ax.errorbar(STATE_RANGE, hc_mean, yerr=hc_sem, fmt='o-',
            color=GROUP_COLORS['hc'], label='HC', lw=2.5, ms=9, capsize=4)

# PD groups
for grp_key in ['tremor_off', 'tremor_on', 'brady_off', 'brady_on']:
    if grp_key not in pd_res or pd_res[grp_key] is None:
        continue
    tll  = pd_res[grp_key]['cv']['test_ll']
    mean = np.nanmean(tll, axis=(0, 2))
    sem  = np.nanstd(tll, axis=(0, 2)) / np.sqrt(tll.shape[0] * tll.shape[2])
    ls   = '-' if 'on' in grp_key else '--'
    ax.errorbar(STATE_RANGE, mean, yerr=sem, fmt=f'o{ls}',
                color=GROUP_COLORS[grp_key], label=GROUP_LABELS[grp_key],
                lw=1.8, ms=7, capsize=3, alpha=0.9)

ax.axvline(CONSENSUS_K, color='#555', ls=':', lw=1.5, label=f'Consensus K={CONSENSUS_K}')
ax.set_xlabel('Number of hidden states (K)', fontsize=13)
ax.set_ylabel('Mean test log-likelihood per trial', fontsize=13)
ax.set_title('Model selection across all groups', fontsize=14)
ax.set_xticks(STATE_RANGE)
ax.legend(fontsize=9, loc='lower right', frameon=False)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(Path(fig_dir, 'fig1_cross_group_model_selection.pdf'), bbox_inches='tight')
plt.show()

## Figure 2 — GLM Weight Heatmaps (5 groups)

In [ ]:
groups_to_plot = [g for g in GROUP_ORDER if g in all_data]
n_groups = len(groups_to_plot)

all_w = np.array([all_data[g]['weights'] for g in groups_to_plot])
vmax  = np.abs(all_w).max()

fig, axes = plt.subplots(1, n_groups, figsize=(3.2 * n_groups, 3 + CONSENSUS_K * 0.7), dpi=DPI)
if n_groups == 1:
    axes = [axes]

for ax, grp in zip(axes, groups_to_plot):
    w = all_data[grp]['weights']  # (K, INPUT_DIM)
    im = ax.imshow(w, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(INPUT_DIM))
    ax.set_xticklabels(MODEL_FEATURES, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(CONSENSUS_K))
    ax.set_yticklabels([f'S{i+1}' for i in range(CONSENSUS_K)], fontsize=10)
    ax.set_title(GROUP_LABELS.get(grp, grp), fontsize=11)
    for i in range(CONSENSUS_K):
        for j in range(INPUT_DIM):
            ax.text(j, i, f'{w[i, j]:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(w[i, j]) > 0.6 * vmax else 'black')

plt.colorbar(im, ax=axes[-1], label='GLM weight', shrink=0.7)
fig.suptitle(f'GLM weights — K={CONSENSUS_K}', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(Path(fig_dir, 'fig2_weight_heatmaps.pdf'), bbox_inches='tight')
plt.show()

## Figure 3 — State Occupancy: 5-Group Bar Plot

Mean ± SEM occupancy per state per group, with individual session dots.

In [ ]:
n_groups    = len(groups_to_plot)
x           = np.arange(n_groups)
bar_width   = 0.75 / CONSENSUS_K
offsets     = np.linspace(-(CONSENSUS_K - 1) * bar_width / 2,
                           (CONSENSUS_K - 1) * bar_width / 2, CONSENSUS_K)

fig, ax = plt.subplots(figsize=(max(8, 1.6 * n_groups), 5), dpi=DPI)

for k in range(CONSENSUS_K):
    means = [all_data[g]['occupancy'][:, k].mean() for g in groups_to_plot]
    sems  = [all_data[g]['occupancy'][:, k].std(ddof=1) / np.sqrt(len(all_data[g]['occupancy']))
             for g in groups_to_plot]
    ax.bar(x + offsets[k], means, width=bar_width, yerr=sems,
           color=STATE_COLORS[k], edgecolor='k', linewidth=0.6,
           capsize=3, error_kw={'elinewidth': 1.2},
           label=f'State {k+1}')
    # Individual session dots
    for g_idx, grp in enumerate(groups_to_plot):
        raw = all_data[grp]['occupancy'][:, k]
        jit = np.random.normal(0, 0.018, len(raw))
        ax.scatter(x[g_idx] + offsets[k] + jit, raw,
                   color='k', s=14, alpha=0.35, zorder=5)

ax.set_xticks(x)
ax.set_xticklabels([GROUP_LABELS.get(g, g) for g in groups_to_plot],
                   fontsize=10, rotation=20, ha='right')
ax.set_ylabel('Fraction of trials in state', fontsize=13)
ax.set_title(f'State occupancy — K={CONSENSUS_K}', fontsize=13)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(Path(fig_dir, 'fig3_state_occupancy.pdf'), bbox_inches='tight')
plt.show()

## Figure 4 — State-specific Psychometric Curves (5 groups)

In [ ]:
x_norm    = np.linspace(-1, 1, 200)
stim_idx  = MODEL_FEATURES.index('normalized_stimulus')
color_idx = MODEL_FEATURES.index('color')
bias_idx  = MODEL_FEATURES.index('bias')

fig, axes = plt.subplots(CONSENSUS_K, 1, figsize=(7, 3.5 * CONSENSUS_K), dpi=DPI, sharex=True)
if CONSENSUS_K == 1:
    axes = [axes]

for k in range(CONSENSUS_K):
    ax = axes[k]
    ax.set_title(f'State {k+1}', fontsize=12)
    for grp in [g for g in GROUP_ORDER if g in all_data]:
        w   = all_data[grp]['weights'][k]
        c   = GROUP_COLORS.get(grp, 'gray')
        lw  = 2.5 if grp == 'hc' else 1.5
        alp = 1.0 if grp == 'hc' else 0.7
        for color_val, ls, cond_lbl in [
            (1.0, '-',  'Unequal prior'),
            (0.0, '--', 'Equal prior'),
        ]:
            logit = w[stim_idx] * x_norm + w[color_idx] * color_val + w[bias_idx]
            lbl   = f'{GROUP_LABELS[grp]}: {cond_lbl}' if k == 0 else None
            ax.plot(x_norm * 100, expit(logit), ls=ls, color=c, lw=lw, alpha=alp, label=lbl)

    ax.axhline(0.5, color='k', lw=0.6, ls=':')
    ax.axvline(0,   color='k', lw=0.6, ls=':')
    ax.set_xlim(-105, 105)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([-100, -35, -13, 0, 13, 35, 100])
    ax.set_ylabel(CHOICE_LABEL, fontsize=10)
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)

axes[0].legend(fontsize=7, ncol=2, frameon=False)
axes[-1].set_xlabel(STIM_LABEL, fontsize=11)
fig.suptitle('State-specific psychometric curves', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(Path(fig_dir, 'fig4_psychometric_curves.pdf'), bbox_inches='tight')
plt.show()

## Figure 5 — Transition Matrices (5 groups)

In [ ]:
fig, axes = plt.subplots(1, n_groups, figsize=(3.0 * n_groups, 3.2), dpi=DPI)
if n_groups == 1:
    axes = [axes]

for ax, grp in zip(axes, groups_to_plot):
    T  = all_data[grp]['transition']
    im = ax.imshow(T, vmin=0, vmax=1, cmap='Blues')
    ax.set_xticks(range(CONSENSUS_K))
    ax.set_yticks(range(CONSENSUS_K))
    ax.set_xticklabels([f'S{i+1}' for i in range(CONSENSUS_K)], fontsize=9)
    ax.set_yticklabels([f'S{i+1}' for i in range(CONSENSUS_K)], fontsize=9)
    ax.set_xlabel('To', fontsize=9)
    ax.set_ylabel('From', fontsize=9)
    ax.set_title(GROUP_LABELS.get(grp, grp), fontsize=11)
    for i in range(CONSENSUS_K):
        for j in range(CONSENSUS_K):
            ax.text(j, i, f'{T[i,j]:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if T[i, j] > 0.6 else 'black')

plt.colorbar(im, ax=axes[-1], label='Probability', shrink=0.85)
fig.suptitle(f'Transition matrices — K={CONSENSUS_K}', fontsize=13)
plt.tight_layout()
plt.savefig(Path(fig_dir, 'fig5_transition_matrices.pdf'), bbox_inches='tight')
plt.show()

## Figure 6 — Prior_strength Weight with Bootstrap CIs

In [ ]:
groups_boot = [g for g in GROUP_ORDER if g in boot_results]
means  = [boot_results[g]['point']  for g in groups_boot]
ci_lo  = [boot_results[g]['ci_lo']  for g in groups_boot]
ci_hi  = [boot_results[g]['ci_hi']  for g in groups_boot]
errs   = [[means[i] - ci_lo[i], ci_hi[i] - means[i]] for i in range(len(means))]

fig, ax = plt.subplots(figsize=(max(6, 1.4 * len(groups_boot)), 4.5), dpi=DPI)
colors_b = [GROUP_COLORS.get(g, 'gray') for g in groups_boot]
x = np.arange(len(groups_boot))
ax.bar(x, means, color=colors_b, edgecolor='k', linewidth=0.8, alpha=0.85)
ax.errorbar(x, means, yerr=np.array(errs).T,
            fmt='none', color='k', capsize=5, capthick=1.5, elinewidth=1.5)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(x)
ax.set_xticklabels([GROUP_LABELS.get(g, g) for g in groups_boot],
                   fontsize=10, rotation=20, ha='right')
ax.set_ylabel('prior_strength weight  (State 1)', fontsize=12)
ax.set_title('Prior integration strength — State 1\n(95% bootstrap CI)', fontsize=12)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig(Path(fig_dir, 'fig6_prior_strength_weights.pdf'), bbox_inches='tight')
plt.show()

## Figure 7 — Paired Medication Effect (ON vs OFF)

In [ ]:
from scipy.stats import wilcoxon as wx

def paired_occupancy(subtype_key, off_key, on_key, k_state=0):
    pairs = subject_pairs[subtype_key]
    off_v, on_v = [], []
    for subj, pair in pairs.items():
        sid_off, sid_on = pair['off'], pair['on']
        off_ids = all_data.get(off_key, {}).get('used_ids', [])
        on_ids  = all_data.get(on_key,  {}).get('used_ids', [])
        if sid_off not in off_ids or sid_on not in on_ids:
            continue
        off_v.append(all_data[off_key]['occupancy'][off_ids.index(sid_off), k_state])
        on_v.append( all_data[on_key]['occupancy'][on_ids.index(sid_on),   k_state])
    return np.array(off_v), np.array(on_v)


med_pairs = [
    ('Tremor',       'tremor', 'tremor_off', 'tremor_on'),
    ('Bradykinesia', 'brady',  'brady_off',  'brady_on'),
]
med_pairs = [(lbl, sk, ok, nk) for lbl, sk, ok, nk in med_pairs
             if ok in all_data and nk in all_data]

if med_pairs:
    fig, axes = plt.subplots(1, len(med_pairs), figsize=(4.5 * len(med_pairs), 5), dpi=DPI)
    if len(med_pairs) == 1:
        axes = [axes]

    for ax, (lbl, sk, off_key, on_key) in zip(axes, med_pairs):
        off_occ, on_occ = paired_occupancy(sk, off_key, on_key, k_state=0)
        n = len(off_occ)
        if n == 0:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center')
            continue

        for ov, nv in zip(off_occ, on_occ):
            ax.plot([0, 1], [ov, nv], color='gray', lw=0.8, alpha=0.5)
        ax.scatter([0]*n, off_occ, color=GROUP_COLORS.get(off_key, 'gray'),
                   s=50, zorder=5, edgecolors='k', lw=0.5)
        ax.scatter([1]*n, on_occ,  color=GROUP_COLORS.get(on_key,  'gray'),
                   s=50, zorder=5, edgecolors='k', lw=0.5)
        for xi, vals, ck in [(0, off_occ, off_key), (1, on_occ, on_key)]:
            ax.errorbar(xi, vals.mean(), yerr=vals.std(ddof=1)/np.sqrt(n),
                        fmt='o', color=GROUP_COLORS.get(ck, 'gray'), ms=12,
                        capsize=5, capthick=2, elinewidth=2, zorder=10,
                        markeredgecolor='k', markeredgewidth=1)

        if n >= 4:
            _, p = wx(off_occ, on_occ, alternative='two-sided')
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
            y_max = max(on_occ.max(), off_occ.max()) + 0.06
            ax.plot([0, 1], [y_max, y_max], 'k', lw=1)
            ax.text(0.5, y_max + 0.01, f'{sig} (p={p:.3f})', ha='center', fontsize=9)

        ax.set_xticks([0, 1])
        ax.set_xticklabels(['OFF', 'ON'], fontsize=12)
        ax.set_ylabel('Bayesian state occupancy', fontsize=11)
        ax.set_title(lbl, fontsize=13)
        ax.set_ylim(-0.02, 1.1)
        for s in ['top', 'right']:
            ax.spines[s].set_visible(False)

    fig.suptitle('State 1 occupancy: ON vs OFF medication\n(paired within subject)', fontsize=13)
    plt.tight_layout()
    plt.savefig(Path(fig_dir, 'fig7_medication_paired.pdf'), bbox_inches='tight')
    plt.show()

## Figure 8 — Trial-Level State Posteriors (Example Sessions)

In [ ]:
example_groups = [g for g in ['hc', 'tremor_off', 'tremor_on', 'brady_off']
                  if g in all_data and len(all_data[g]['viterbi']) > 0]
n_rows = CONSENSUS_K + 1

fig, axes = plt.subplots(n_rows, len(example_groups),
                          figsize=(5 * len(example_groups), 2.5 + n_rows * 1.2),
                          sharex=False, dpi=DPI)
if len(example_groups) == 1:
    axes = axes.reshape(-1, 1)

for col, grp in enumerate(example_groups):
    vit  = all_data[grp]['viterbi'][0]
    post = all_data[grp]['smoothed'][0]
    T    = len(vit)

    axes[0, col].imshow(vit.reshape(1, -1), aspect='auto',
                        cmap='tab10', vmin=0, vmax=CONSENSUS_K - 1, interpolation='none')
    axes[0, col].set_yticks([0])
    axes[0, col].set_yticklabels(['Viterbi'], fontsize=8)
    axes[0, col].set_title(GROUP_LABELS.get(grp, grp), fontsize=10)

    for k in range(CONSENSUS_K):
        axes[k+1, col].fill_between(range(T), post[:, k],
                                    color=STATE_COLORS[k], alpha=0.7)
        axes[k+1, col].set_ylim(0, 1)
        axes[k+1, col].set_yticks([0, 1])
        axes[k+1, col].set_ylabel(f'P(S{k+1})', fontsize=8)
        for s in ['top', 'right']:
            axes[k+1, col].spines[s].set_visible(False)
    axes[-1, col].set_xlabel('Trial', fontsize=9)

fig.suptitle('Example sessions: trial-level state posteriors', fontsize=12)
plt.tight_layout()
plt.savefig(Path(fig_dir, 'fig8_example_posteriors.pdf'), bbox_inches='tight')
plt.show()

## Figure 9 — Dwell Time Distributions

In [ ]:
def compute_dwell_times(viterbi_list, k_state):
    dwells = []
    for vit in viterbi_list:
        count = 0
        for s in vit:
            if s == k_state:
                count += 1
            else:
                if count > 0:
                    dwells.append(count)
                count = 0
        if count > 0:
            dwells.append(count)
    return np.array(dwells)


fig, axes = plt.subplots(1, CONSENSUS_K, figsize=(4.5 * CONSENSUS_K, 4), dpi=DPI)
if CONSENSUS_K == 1:
    axes = [axes]

for k in range(CONSENSUS_K):
    ax = axes[k]
    for grp in groups_to_plot:
        d = compute_dwell_times(all_data[grp]['viterbi'], k)
        if len(d) == 0:
            continue
        ax.hist(d, bins=np.arange(0.5, 51.5), density=True, alpha=0.55,
                color=GROUP_COLORS.get(grp, 'gray'), label=GROUP_LABELS.get(grp, grp))
    ax.set_title(f'State {k+1}', fontsize=11)
    ax.set_xlabel('Dwell time (trials)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_xlim(0, 50)
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    if k == 0:
        ax.legend(fontsize=7, frameon=False)

fig.suptitle('Dwell time distributions per hidden state', fontsize=13)
plt.tight_layout()
plt.savefig(Path(fig_dir, 'fig9_dwell_times.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
import os
print(f'Figures saved to: {fig_dir}')
for f in sorted(os.listdir(fig_dir)):
    print(f'  {f}')